# 탑뷰와 경유지를 함께 배우는 PPO 학습
로컬 소스 ZIP 또는 본인이 준비한 GitHub 저장소를 불러옵니다. 기존 인식 ONNX나 데이터셋은 필요 없습니다. **런타임 → 런타임 유형 변경 → GPU**를 선택한 뒤 위에서 아래로 실행하세요.

최근 4장의 RGB와 공개 좌표를 입력합니다. 좌표 추종 제어기의 기본 행동에 신경망이 보정을 더하며, 탑뷰 CNN도 PPO로 함께 갱신합니다. 0단계부터 시작하고 결과를 확인한 뒤 복잡도를 올립니다. GPU는 신경망을 가속하지만 시뮬레이션·RGB 렌더링은 CPU에서 수행합니다.

In [ ]:
import os, sys, subprocess
from pathlib import Path
import torch
print('Python:', sys.version.split()[0])
print('Torch:', torch.__version__, 'CUDA runtime:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available(), 'CPU count:', os.cpu_count())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    DEVICE = 'cuda'
else:
    print('GPU를 선택한 새 런타임에서 실행하면 CUDA를 사용할 수 있습니다. 현재는 CPU를 사용합니다.')
    DEVICE = 'cpu'


## 패키지 설치
Colab에 설치된 Torch를 유지합니다. 로컬의 CPU 전용 requirements 파일은 사용하지 않습니다. 설치 충돌 시 새 GPU 런타임에서 다시 시작하세요.


In [ ]:
import importlib.metadata as metadata
torch_pin = Path('/content/colab-torch-constraint.txt')
torch_pin.write_text('torch==' + metadata.version('torch') + '\n')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-c', str(torch_pin),
                'numpy>=2,<3', 'gymnasium==1.2.3', 'stable-baselines3==2.9.0',
                'matplotlib>=3.9,<4'], check=True)
for name in ('numpy', 'gymnasium', 'stable-baselines3', 'torch', 'matplotlib'):
    print(name, metadata.version(name))


## 소스 불러오기
기본은 ZIP 업로드입니다. 로컬에서 `python -m training.package_colab`로 만든 소스 ZIP을 선택하세요. GitHub를 쓰려면 아래 모드를 `github`으로 바꾸고 실제 저장소 주소를 입력하세요. 현재 폴더가 GitHub에 올라가 있다고 가정하지 않습니다.
이 셀은 새 `/content/usvnav_project` 폴더를 사용합니다. 같은 런타임에서 다시 로드할 때는 새 `PROJECT_ROOT`를 지정하거나 기존 불러오기 셀을 건너뛰세요.


In [ ]:
SOURCE_MODE = 'zip'  # 'zip' 또는 'github'
GITHUB_URL = 'https://github.com/YOUR_ACCOUNT/YOUR_REPOSITORY.git'
GITHUB_REF = ''  # 기본 브랜치는 빈 문자열; 특정 브랜치나 태그를 쓰려면 입력
PROJECT_ROOT = Path('/content/usvnav_project')
if PROJECT_ROOT.exists():
    raise FileExistsError('이미 소스 폴더가 있습니다. 새 PROJECT_ROOT를 지정하거나 이 셀을 건너뛰세요.')
if SOURCE_MODE == 'zip':
    from google.colab import files
    import hashlib, io, json, stat, zipfile
    from pathlib import PurePosixPath
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('소스 ZIP 하나만 선택하세요.')
    archive_name, archive_bytes = next(iter(uploaded.items()))
    with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
        manifest = json.loads(archive.read('_source_manifest.json'))
        if manifest.get('format') != 'usvnav-colab-source/1':
            raise ValueError('training.package_colab에서 생성한 ZIP을 사용하세요.')
        expected = set(manifest['files']) | {'_source_manifest.json'}
        if set(archive.namelist()) != expected or len(archive.namelist()) != len(expected):
            raise ValueError('ZIP 파일 목록이 manifest와 다릅니다.')
        if sum(item.file_size for item in archive.infolist()) > 100_000_000:
            raise ValueError('소스 ZIP의 해제 크기가 너무 큽니다.')
        for item in archive.infolist():
            name = item.filename
            if '\\' in name or ':' in name or PurePosixPath(name).is_absolute() or '..' in PurePosixPath(name).parts:
                raise ValueError('잘못된 ZIP 경로: ' + name)
            if stat.S_ISLNK(item.external_attr >> 16):
                raise ValueError('심볼릭 링크는 허용하지 않습니다.')
            if name != '_source_manifest.json':
                payload = archive.read(name)
                record = manifest['files'][name]
                if len(payload) != record['bytes'] or hashlib.sha256(payload).hexdigest() != record['sha256']:
                    raise ValueError('파일 검증 실패: ' + name)
        PROJECT_ROOT.mkdir(parents=True)
        archive.extractall(PROJECT_ROOT)
    print('Verified source ZIP:', archive_name, 'SHA-256:', hashlib.sha256(archive_bytes).hexdigest())
elif SOURCE_MODE == 'github':
    from urllib.parse import urlparse
    parsed = urlparse(GITHUB_URL)
    if parsed.scheme != 'https' or parsed.hostname != 'github.com' or parsed.username or parsed.password or parsed.query or parsed.fragment:
        raise ValueError('토큰 없는 https://github.com/소유자/저장소.git 주소를 입력하세요.')
    if 'YOUR_' in GITHUB_URL or len(parsed.path.strip('/').split('/')) != 2:
        raise ValueError('본인의 실제 저장소 주소로 바꿔주세요.')
    clone = ['git', 'clone', '--depth', '1']
    if GITHUB_REF:
        clone += ['--branch', GITHUB_REF]
    subprocess.run(clone + [GITHUB_URL, str(PROJECT_ROOT)], check=True)
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], check=True)
else:
    raise ValueError('SOURCE_MODE는 zip 또는 github입니다.')
if not (PROJECT_ROOT / 'training' / 'train_rgb_navigation.py').is_file():
    raise FileNotFoundError('RGB PPO 학습 코드가 포함된 최신 소스를 사용하세요.')
os.chdir(PROJECT_ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', str(PROJECT_ROOT)], check=True)
subprocess.run([sys.executable, '-B', '-m', 'training.train_rgb_navigation', '--help'], check=True)


## Google Drive에 결과 저장
아래 셀을 직접 실행하고 본인 Google 계정으로 승인합니다. 코드와 시뮬레이션은 `/content`에서 실행하고, 결과와 체크포인트는 Drive에 저장합니다. Colab 런타임 파일은 세션 종료 후 사라질 수 있습니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUNS_ROOT = Path('/content/drive/MyDrive/usvnav-rgb-ppo')
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
print('Checkpoints and reports:', RUNS_ROOT)
from datetime import datetime, timezone
from uuid import uuid4
def new_output(label):
    return RUNS_ROOT / (label + '-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid4().hex[:6])
def run_training(output, steps, stage=0, n_envs=1, resume=None, seed=7):
    command = [sys.executable, '-u', '-B', '-m', 'training.train_rgb_navigation',
               '--out', str(output), '--steps', str(steps), '--stage', str(stage),
               '--n-envs', str(n_envs), '--device', DEVICE, '--seed', str(seed)]
    if resume is not None:
        command += ['--resume', str(resume)]
    print(' '.join(command), flush=True)
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
    print('Saved:', output)


## 선택: 로컬에서 학습한 정책 업로드
최신 로컬 실험의 `training/runs/rgb-ppo-stage1-02/final-policy.zip`를 이어 학습하려면 아래 값을 `True`로 바꾸고 해당 ZIP만 선택하세요. 이 정책은 1단계까지 학습했으므로 `LOCAL_CHECKPOINT_STAGE=1`을 사용하고, 학습 셀도 1단계로 설정됩니다. 0단계 정책을 업로드한다면 이 값을 0으로 바꾸세요. 소스 ZIP에는 체크포인트가 포함되지 않습니다. 업로드하지 않으면 아래 실행 검사 정책으로 0단계를 이어 학습합니다. 본인이 생성한 체크포인트를 사용하세요.


In [ ]:
UPLOAD_LOCAL_CHECKPOINT = False
LOCAL_CHECKPOINT_STAGE = 1  # 최신 rgb-ppo-stage1-02 정책은 1; 다른 정책이면 그 단계에 맞게 변경
LOCAL_CHECKPOINT = None
if UPLOAD_LOCAL_CHECKPOINT:
    from google.colab import files
    import io, zipfile
    uploaded_policy = files.upload()
    if len(uploaded_policy) != 1:
        raise ValueError('학습한 정책 ZIP 하나만 선택하세요.')
    uploaded_name, uploaded_bytes = next(iter(uploaded_policy.items()))
    with zipfile.ZipFile(io.BytesIO(uploaded_bytes)) as policy_archive:
        if not {'data', 'policy.pth'}.issubset(policy_archive.namelist()):
            raise ValueError('SB3 정책 체크포인트 ZIP을 선택하세요. 소스 ZIP과 다릅니다.')
    upload_folder = new_output('uploaded-local-policy')
    upload_folder.mkdir(parents=True)
    LOCAL_CHECKPOINT = upload_folder / 'local-final-policy.zip'
    LOCAL_CHECKPOINT.write_bytes(uploaded_bytes)
    print('Local policy saved:', uploaded_name, '→', LOCAL_CHECKPOINT)
else:
    print('로컬 정책 업로드를 생략합니다. 실행 검사에서 학습한 정책을 이어 받습니다.')


## 먼저 짧은 실행 검사
환경 생성, RGB CNN 역전파, PPO 갱신, 저장이 연결되는지 확인합니다. 이 길이로 주행 능력 확보를 판단하지 않습니다. 실제 수집량은 rollout 묶음 단위로 늘어날 수 있습니다.


In [ ]:
SMOKE_OUT = new_output('smoke-stage0')
run_training(SMOKE_OUT, steps=256, stage=0, n_envs=1)
print('Short run files:', [p.name for p in SMOKE_OUT.iterdir()])


## 주행 학습 이어하기
초기값은 추가 10,000회 의사결정입니다. 로컬 정책을 업로드했다면 지정한 단계에서 그 정책을, 아니면 위 실행 검사 정책으로 0단계를 이어 학습합니다. 처음에는 실행 속도를 보고 예산을 조정하세요. 환경 수는 CPU 할당에 맞게 1~2개부터 비교합니다. GPU 사용률이 낮다면 경험 생성이 병목일 수 있습니다. 0단계는 장애물 없는 직선 추종, 1단계는 방향 전환, 2단계는 부표 하나 회피입니다. 2단계는 1단계와 이전 단계의 검증을 더 확인한 뒤 선택하세요.


In [ ]:
STAGE = LOCAL_CHECKPOINT_STAGE if LOCAL_CHECKPOINT is not None else 0
STEPS = 10_000
N_ENVS = max(1, min(2, os.cpu_count() or 1))
TRAIN_OUT = new_output(f'train-stage{STAGE}')
TRAIN_CHECKPOINT = LOCAL_CHECKPOINT if LOCAL_CHECKPOINT is not None else SMOKE_OUT / 'final-policy.zip'
if not TRAIN_CHECKPOINT.is_file():
    raise FileNotFoundError('로컬 정책 업로드 또는 위 실행 검사를 먼저 완료하세요: ' + str(TRAIN_CHECKPOINT))
print('Continue policy:', TRAIN_CHECKPOINT)
run_training(TRAIN_OUT, steps=STEPS, stage=STAGE, n_envs=N_ENVS, resume=TRAIN_CHECKPOINT)


## 선택: 체크포인트에서 추가 학습
이 셀은 경로를 입력할 때만 실행합니다. 최신 체크포인트를 Drive 파일 목록에서 확인하세요. `--steps`는 이번 실행에서 추가할 수집량입니다. 정책·최적화 상태를 이어 받지만, 환경의 직전 위치·난수 상태까지 같은 궤적으로 이어지는 것은 아닙니다.


In [ ]:
RESUME_CHECKPOINT = ''  # 예: /content/drive/MyDrive/usvnav-rgb-ppo/train-stage0-.../latest-policy.zip
RESUME_STAGE = STAGE  # 위에서 선택한 단계; 검증 후 다음 단계로 올리려면 변경
ADDITIONAL_STEPS = 10_000
if RESUME_CHECKPOINT:
    checkpoint = Path(RESUME_CHECKPOINT)
    if not checkpoint.is_file():
        raise FileNotFoundError(checkpoint)
    RESUME_OUT = new_output(f'resume-stage{RESUME_STAGE}')
    run_training(RESUME_OUT, steps=ADDITIONAL_STEPS, stage=RESUME_STAGE,
                 n_envs=N_ENVS, resume=checkpoint)
else:
    print('재개하려면 RESUME_CHECKPOINT에 실제 ZIP 파일 경로를 입력하세요.')


Colab의 GPU 종류·접속 가능 여부·실행 한도는 변동됩니다. 세션 중 체크포인트를 저장하고, 끊기면 위 재개 셀을 사용하세요.
[공식 Colab FAQ](https://research.google.com/colaboratory/faq.html) · [공식 파일 입출력 예제](https://colab.research.google.com/notebooks/io.ipynb)
이 노트북은 로컬에서 JSON·Python 문법을 검사했습니다. Google 계정 승인과 실제 Colab GPU 학습은 이 노트북을 실행하여 확인해야 합니다.